# 04. BERT vs GPT: Transformer 아키텍처 변형

## 학습 목표
- Encoder-only (BERT), Decoder-only (GPT), Encoder-Decoder (T5) 차이 이해
- 각 아키텍처의 학습 목표(MLM vs CLM) 이해
- Attention mask 패턴 차이 시각화
- HuggingFace로 BERT, GPT-2 실습
- 현대 LLM이 Decoder-only를 선택한 이유

## 핵심 논문
- [BERT](https://arxiv.org/abs/1810.04805) (Devlin et al., 2019)
- [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (Radford et al., 2019)
- [T5](https://arxiv.org/abs/1910.10683) (Raffel et al., 2020)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

# HuggingFace transformers (pip install transformers)
# Google Colab에서는 기본 설치되어 있음
try:
    from transformers import (
        BertTokenizer, BertForMaskedLM,
        GPT2Tokenizer, GPT2LMHeadModel,
        pipeline
    )
    HF_AVAILABLE = True
    print("HuggingFace transformers 사용 가능")
except ImportError:
    HF_AVAILABLE = False
    print("transformers 미설치. pip install transformers 실행 필요")
    print("HuggingFace 실습 셀은 건너뛸 수 있습니다.")

torch.manual_seed(42)

## 1. 세 가지 Transformer 아키텍처

| | Encoder-only | Decoder-only | Encoder-Decoder |
|---|-------------|-------------|----------------|
| 대표 모델 | **BERT** | **GPT** | **T5, BART** |
| Attention | 양방향 (Bidirectional) | 단방향 (Causal) | 양방향 + 단방향 |
| 학습 목표 | MLM (빈칸 채우기) | CLM (다음 토큰 예측) | Seq2Seq |
| 강점 | 이해 (NLU) | 생성 (NLG) | 이해 + 생성 |
| 예시 task | 분류, NER, QA | 텍스트 생성, 챗봇 | 번역, 요약 |

In [ ]:
# 세 가지 아키텍처의 구조 시각화

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 공통 설정
seq_len = 6
tokens = ['[CLS]', 'I', 'love', 'NLP', '.', '[SEP]']

# 1. BERT (Encoder-only): 양방향
ax = axes[0]
mask_bert = np.ones((seq_len, seq_len))
ax.imshow(mask_bert, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=8, rotation=45)
ax.set_yticklabels(tokens, fontsize=8)
ax.set_title('BERT (Encoder-only)\nBidirectional Attention', fontsize=12)
ax.set_xlabel('Key')
ax.set_ylabel('Query')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, '1', ha='center', va='center', fontsize=9)

# 2. GPT (Decoder-only): 단방향
ax = axes[1]
gpt_tokens = ['I', 'love', 'NLP', 'and', 'ML', '.']
mask_gpt = np.tril(np.ones((seq_len, seq_len)))
ax.imshow(mask_gpt, cmap='Oranges', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(gpt_tokens, fontsize=8, rotation=45)
ax.set_yticklabels(gpt_tokens, fontsize=8)
ax.set_title('GPT (Decoder-only)\nCausal Attention', fontsize=12)
ax.set_xlabel('Key')
ax.set_ylabel('Query')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{int(mask_gpt[i,j])}', ha='center', va='center',
                fontsize=9, color='white' if mask_gpt[i,j] > 0.5 else 'black')

# 3. T5 (Encoder-Decoder): 양방향(Enc) + 단방향(Dec) + Cross
ax = axes[2]
enc_len = 3
dec_len = 3
mask_t5 = np.zeros((seq_len, seq_len))
mask_t5[:enc_len, :enc_len] = 1                     # Encoder: 양방향
mask_t5[enc_len:, :enc_len] = 0.6                   # Cross-Attention
mask_t5[enc_len:, enc_len:] = np.tril(np.ones((dec_len, dec_len)))  # Decoder: 단방향

t5_tokens = ['나는', '학생', '이다', 'I', 'am', 'student']
ax.imshow(mask_t5, cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(t5_tokens, fontsize=8, rotation=45)
ax.set_yticklabels(t5_tokens, fontsize=8)
ax.set_title('T5 (Encoder-Decoder)\nBidir + Causal + Cross', fontsize=12)
ax.set_xlabel('Key (Encoder | Decoder)')
ax.set_ylabel('Query (Encoder | Decoder)')
ax.axvline(x=enc_len - 0.5, color='red', linestyle='--', alpha=0.7)
ax.axhline(y=enc_len - 0.5, color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

print("BERT: 모든 토큰이 서로 볼 수 있음 (양방향)")
print("GPT:  각 토큰이 이전 토큰만 볼 수 있음 (단방향)")
print("T5:   Encoder는 양방향, Decoder는 단방향 + Encoder를 봄")

---
## 2. Encoder-only: BERT

### 핵심 특징
- **양방향** (Bidirectional): 모든 토큰이 모든 토큰을 볼 수 있음
- **MLM** (Masked Language Model): 입력의 일부를 `[MASK]`로 가리고 예측
- **[CLS] 토큰**: 문장 전체 표현, 분류 task에 사용

### MLM 학습
```
입력: [CLS] 나는 [MASK] 좋아한다 [SEP]
예측: [MASK] → "커피를"
```

- 전체 토큰의 15%를 마스킹
  - 80%: `[MASK]`로 대체
  - 10%: 랜덤 토큰으로 대체
  - 10%: 그대로 유지

In [ ]:
# MLM 학습 과정 시뮬레이션

def simulate_mlm(sentence_tokens, mask_prob=0.15):
    """BERT의 MLM 전처리 시뮬레이션"""
    tokens = sentence_tokens.copy()
    labels = [-100] * len(tokens)  # -100 = 손실 계산 제외
    
    for i in range(len(tokens)):
        if tokens[i] in ['[CLS]', '[SEP]']:
            continue  # 특수 토큰은 건너뜀
        
        if np.random.random() < mask_prob:
            labels[i] = tokens[i]  # 원래 토큰을 정답으로
            
            r = np.random.random()
            if r < 0.8:
                tokens[i] = '[MASK]'   # 80%: 마스크
            elif r < 0.9:
                tokens[i] = f'<rand>'  # 10%: 랜덤 토큰
            # else: 10% 그대로 유지
    
    return tokens, labels


np.random.seed(42)
original = ['[CLS]', '오늘', '날씨', '가', '정말', '좋다', '[SEP]']

print(f"원본:    {original}")
for i in range(3):
    masked, labels = simulate_mlm(original, mask_prob=0.3)  # 시각화를 위해 높은 비율
    print(f"마스킹 {i+1}: {masked}")
    label_str = [l if l != -100 else '-' for l in labels]
    print(f"정답:    {label_str}")
    print()

---
## 3. Decoder-only: GPT

### 핵심 특징
- **단방향** (Unidirectional/Causal): 이전 토큰만 볼 수 있음
- **CLM** (Causal Language Model): 다음 토큰을 예측
- **자기회귀** (Autoregressive): 한 토큰씩 순차적으로 생성

### CLM 학습
```
입력: 나는 커피를 좋아
정답: 커피를 좋아 한다

위치 1: "나는" → 다음 토큰 "커피를" 예측
위치 2: "나는 커피를" → 다음 토큰 "좋아" 예측
위치 3: "나는 커피를 좋아" → 다음 토큰 "한다" 예측
```

In [ ]:
# CLM (Causal Language Modeling) 학습 과정 시뮬레이션

tokens = ["나는", "커피를", "좋아", "한다", "."]

print("=== GPT의 CLM 학습 ===")
print(f"전체 시퀀스: {tokens}\n")

for i in range(len(tokens) - 1):
    context = tokens[:i+1]
    target = tokens[i+1]
    print(f"Step {i+1}:")
    print(f"  입력 (context): {context}")
    print(f"  예측 (target):  '{target}'")
    print(f"  Causal mask: 뒤의 {len(tokens)-i-2}개 토큰은 볼 수 없음")
    print()

print("→ 각 위치에서 '다음 토큰'을 예측하는 것이 학습 목표")
print("→ 추론 시에도 한 토큰씩 순차적으로 생성 (autoregressive)")

---
## 4. Encoder-Decoder: T5, BART

### 핵심 특징
- Encoder: 입력을 양방향으로 이해
- Decoder: Encoder의 표현을 참조하면서 출력을 자기회귀적으로 생성
- Cross-Attention: Decoder가 Encoder 출력에 attend

### T5의 접근: 모든 NLP task를 text-to-text로

```
번역:    "translate English to Korean: Hello" → "안녕하세요"
요약:    "summarize: (긴 텍스트)" → "요약 결과"
분류:    "classify: I love this movie" → "positive"
```

In [ ]:
# 세 아키텍처의 Attention 패턴 비교

def visualize_attention_patterns():
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # BERT: 양방향 Self-Attention
    ax = axes[0]
    bert_mask = np.ones((5, 5))
    ax.imshow(bert_mask, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_title('BERT\nSelf-Attention', fontsize=12)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, 'O', ha='center', va='center', fontsize=12, color='white')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')
    
    # GPT: Causal Self-Attention
    ax = axes[1]
    gpt_mask = np.tril(np.ones((5, 5)))
    ax.imshow(gpt_mask, cmap='Oranges', vmin=0, vmax=1, aspect='auto')
    ax.set_title('GPT\nCausal Self-Attention', fontsize=12)
    for i in range(5):
        for j in range(5):
            symbol = 'O' if gpt_mask[i,j] else 'X'
            color = 'white' if gpt_mask[i,j] else 'gray'
            ax.text(j, i, symbol, ha='center', va='center', fontsize=12, color=color)
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')
    
    # T5 Encoder: 양방향
    ax = axes[2]
    t5_enc_mask = np.ones((5, 5))
    ax.imshow(t5_enc_mask, cmap='Greens', vmin=0, vmax=1, aspect='auto')
    ax.set_title('T5 Encoder\nBidirectional', fontsize=12)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, 'O', ha='center', va='center', fontsize=12, color='white')
    ax.set_xlabel('Key (src)')
    ax.set_ylabel('Query (src)')
    
    # T5 Decoder: Causal + Cross
    ax = axes[3]
    # 4개 디코더 토큰이 5개 인코더 토큰을 봄
    t5_cross = np.ones((4, 5))  # Cross: 모든 인코더 토큰 참조
    ax.imshow(t5_cross, cmap='Purples', vmin=0, vmax=1, aspect='auto')
    ax.set_title('T5 Decoder\nCross-Attention', fontsize=12)
    for i in range(4):
        for j in range(5):
            ax.text(j, i, 'O', ha='center', va='center', fontsize=12, color='white')
    ax.set_xlabel('Key (encoder output)')
    ax.set_ylabel('Query (decoder)')
    
    plt.tight_layout()
    plt.show()

visualize_attention_patterns()

print("O = attend 가능, X = attend 불가")
print("\nBERT: 모든 위치가 서로 참조 가능 → '이해'에 유리")
print("GPT: 현재 위치 이전만 참조 가능 → '생성'에 적합")
print("T5: Encoder는 양방향 이해, Decoder는 Encoder를 참조하며 생성")

---
## 5. HuggingFace로 BERT 실습: MLM

BERT의 핵심 기능: 마스킹된 토큰을 문맥에서 예측

In [ ]:
if HF_AVAILABLE:
    # BERT MLM 파이프라인
    fill_mask = pipeline('fill-mask', model='bert-base-uncased')
    
    # 빈칸 채우기
    sentences = [
        "The capital of France is [MASK].",
        "I want to [MASK] a book at the library.",
        "Machine [MASK] is a branch of artificial intelligence.",
    ]
    
    for sent in sentences:
        print(f"\n입력: {sent}")
        results = fill_mask(sent)
        for r in results[:3]:  # Top 3
            print(f"  예측: {r['token_str']:>12} (score: {r['score']:.4f})")
else:
    print("HuggingFace 미설치. 아래는 예상 결과입니다:")
    print()
    print('입력: The capital of France is [MASK].')
    print('  예측:        paris (score: 0.8812)')
    print('  예측:       london (score: 0.0134)')
    print('  예측:     lyon     (score: 0.0061)')
    print()
    print('입력: I want to [MASK] a book at the library.')
    print('  예측:       read  (score: 0.3842)')
    print('  예측:       buy   (score: 0.2156)')
    print('  예측:       borrow (score: 0.1543)')

In [ ]:
if HF_AVAILABLE:
    # BERT 문장 임베딩 (CLS 토큰)
    from transformers import BertTokenizer, BertModel
    
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model_bert = BertModel.from_pretrained('bert-base-uncased')
    model_bert.eval()
    
    sentences = [
        "I love machine learning.",
        "I enjoy deep learning.",
        "The weather is nice today.",
        "It is sunny outside.",
    ]
    
    embeddings = []
    with torch.no_grad():
        for sent in sentences:
            inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True)
            outputs = model_bert(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] 토큰
            embeddings.append(cls_embedding.squeeze())
    
    # 문장 간 코사인 유사도
    print("문장 간 코사인 유사도:")
    for i in range(len(sentences)):
        for j in range(i+1, len(sentences)):
            sim = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0))
            print(f"  '{sentences[i][:30]}' vs '{sentences[j][:30]}': {sim.item():.4f}")
else:
    print("HuggingFace 미설치. 아래는 예상 결과입니다:")
    print()
    print("문장 간 코사인 유사도:")
    print("  'I love machine learning.' vs 'I enjoy deep learning.': 0.9234")
    print("  'I love machine learning.' vs 'The weather is nice today.': 0.4521")
    print("  'I love machine learning.' vs 'It is sunny outside.': 0.4103")
    print("  'I enjoy deep learning.' vs 'The weather is nice today.': 0.4312")
    print("  'I enjoy deep learning.' vs 'It is sunny outside.': 0.3987")
    print("  'The weather is nice today.' vs 'It is sunny outside.': 0.8876")
    print("\n→ 의미가 비슷한 문장일수록 유사도가 높다")

---
## 6. HuggingFace로 GPT-2 실습: 텍스트 생성

GPT의 핵심 기능: 주어진 프롬프트에서 이어지는 텍스트를 자기회귀적으로 생성

In [ ]:
if HF_AVAILABLE:
    # GPT-2 텍스트 생성
    generator = pipeline('text-generation', model='gpt2')
    
    prompts = [
        "Artificial intelligence will",
        "The best way to learn programming is",
        "In the future, robots will",
    ]
    
    for prompt in prompts:
        print(f"\n프롬프트: '{prompt}'")
        results = generator(prompt, max_length=40, num_return_sequences=2,
                           do_sample=True, temperature=0.7, pad_token_id=50256)
        for i, r in enumerate(results):
            print(f"  생성 {i+1}: {r['generated_text']}")
else:
    print("HuggingFace 미설치. 아래는 예상 결과입니다:")
    print()
    print("프롬프트: 'Artificial intelligence will'")
    print("  생성 1: Artificial intelligence will transform how we work and live,")
    print("          enabling machines to perform tasks previously thought impossible.")
    print("  생성 2: Artificial intelligence will be a key driver of economic growth")
    print("          in the coming decades, according to a new report.")

In [ ]:
# GPT-2의 토큰별 생성 과정 시각화

if HF_AVAILABLE:
    tokenizer_gpt = GPT2Tokenizer.from_pretrained('gpt2')
    model_gpt = GPT2LMHeadModel.from_pretrained('gpt2')
    model_gpt.eval()
    
    prompt = "The meaning of life is"
    input_ids = tokenizer_gpt.encode(prompt, return_tensors='pt')
    
    print(f"프롬프트: '{prompt}'")
    print(f"토큰 ID: {input_ids.tolist()}")
    print(f"\n토큰별 생성 과정:")
    
    generated = input_ids
    for step in range(10):
        with torch.no_grad():
            outputs = model_gpt(generated)
            next_token_logits = outputs.logits[:, -1, :]  # 마지막 위치의 logits
            probs = F.softmax(next_token_logits, dim=-1)
            
            # Top 5 후보
            top5 = torch.topk(probs, 5, dim=-1)
            
            # Greedy: 가장 확률 높은 토큰 선택
            next_token = top5.indices[0, 0].unsqueeze(0).unsqueeze(0)
            generated = torch.cat([generated, next_token], dim=1)
            
            token_str = tokenizer_gpt.decode(next_token.squeeze())
            candidates = [(tokenizer_gpt.decode(top5.indices[0, i]), top5.values[0, i].item())
                          for i in range(5)]
            cand_str = ', '.join([f"'{t}'({p:.2%})" for t, p in candidates])
            print(f"  Step {step+1}: 선택='{token_str}' | 후보: {cand_str}")
    
    print(f"\n최종: {tokenizer_gpt.decode(generated.squeeze())}")
else:
    print("HuggingFace 미설치. 아래는 예상 결과입니다:")
    print()
    print("프롬프트: 'The meaning of life is'")
    print("  Step 1: 선택=' to' | 후보: ' to'(15.23%), ' a'(8.12%), ' not'(6.54%), ...")
    print("  Step 2: 선택=' be' | 후보: ' be'(22.31%), ' find'(5.67%), ' live'(4.89%), ...")
    print("  ...")
    print("\n→ 매 step마다 전체 vocab에 대한 확률 분포를 계산하고, 토큰을 선택")

---
## 7. 왜 현대 LLM은 Decoder-only를 선택했는가?

GPT-3, GPT-4, LLaMA, Claude, Gemini 등 대부분의 최신 LLM은 **Decoder-only** 아키텍처.

### 이유 1: 학습 효율성
- CLM은 모든 토큰이 학습 신호 → 데이터 효율적
- MLM은 15%만 학습 신호 → 상대적으로 비효율

### 이유 2: 스케일링 단순성
- Decoder-only는 구조가 단순 (하나의 스택)
- Encoder-Decoder는 두 개의 스택 → 하이퍼파라미터 2배

### 이유 3: 통합 인터페이스
- "모든 것은 다음 토큰 예측"
- 분류, 생성, 번역, 추론 등을 하나의 프레임워크로

### 이유 4: In-context Learning
- 프롬프트에 예시를 넣으면 fine-tuning 없이 task 수행
- BERT는 이것이 불가능

### 이유 5: KV Cache
- 자기회귀 생성 시 이전 토큰의 K, V를 캐시
- Encoder-Decoder보다 추론 최적화가 쉬움

In [ ]:
# 학습 효율성 비교: MLM vs CLM

sequence = ["나는", "오늘", "카페에서", "커피를", "마셨다"]
seq_len = len(sequence)

print("=== MLM (BERT) ===")
print(f"시퀀스: {sequence}")
mlm_masked = sequence.copy()
mlm_masked[2] = '[MASK]'  # 15% 마스킹 (5개 중 1개)
print(f"마스킹: {mlm_masked}")
print(f"학습 신호: 1/{seq_len} = {1/seq_len:.0%} 의 토큰에서만 손실 계산")

print(f"\n=== CLM (GPT) ===")
print(f"시퀀스: {sequence}")
for i in range(seq_len - 1):
    context = sequence[:i+1]
    target = sequence[i+1]
    print(f"  {context} → '{target}'")
print(f"학습 신호: {seq_len-1}/{seq_len} = {(seq_len-1)/seq_len:.0%} 의 토큰에서 손실 계산")

print(f"\n→ CLM이 같은 데이터에서 더 많은 학습 신호를 얻는다")
print(f"→ 대규모 학습에서 이 차이가 크게 작용")

In [ ]:
# 아키텍처별 적합한 Task 정리

data = {
    'Task': ['감정 분석', '개체명 인식', '텍스트 생성', '챗봇/대화',
             '번역', '요약', '질의응답', 'In-context Learning'],
    'BERT\n(Encoder)': ['★★★', '★★★', '★', '★', '★', '★', '★★', '★'],
    'GPT\n(Decoder)': ['★★', '★★', '★★★', '★★★', '★★', '★★', '★★', '★★★'],
    'T5\n(Enc-Dec)': ['★★★', '★★', '★★', '★★', '★★★', '★★★', '★★★', '★★'],
}

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('tight')
ax.axis('off')

table = ax.table(
    cellText=list(zip(data['Task'], data['BERT\n(Encoder)'], data['GPT\n(Decoder)'], data['T5\n(Enc-Dec)'])),
    colLabels=['Task', 'BERT (Encoder)', 'GPT (Decoder)', 'T5 (Enc-Dec)'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

# 헤더 색상
for j in range(4):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')

plt.title('Architecture vs Task Suitability', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: MLM vs CLM 손실 함수 비교

같은 시퀀스에 대해 MLM과 CLM의 손실(loss)을 직접 계산하고 비교하세요.

- vocab_size = 10, seq_len = 6
- 랜덤 logits와 target으로 Cross-Entropy Loss 계산
- MLM: 15%만 마스킹하여 해당 위치만 loss 계산
- CLM: 모든 위치에서 다음 토큰 예측 loss 계산
- 두 방식의 유효 학습 토큰 수를 비교

In [ ]:
torch.manual_seed(42)
vocab_size = 10
seq_len = 20
batch_size = 4

# 랜덤 logits과 target
logits = torch.randn(batch_size, seq_len, vocab_size)
targets = torch.randint(0, vocab_size, (batch_size, seq_len))

# TODO: MLM loss 계산 (15% 마스킹)
# 1. 랜덤으로 15% 위치를 선택
# 2. 해당 위치만 CrossEntropyLoss 계산
# mlm_loss = ...

# TODO: CLM loss 계산 (모든 위치)
# 1. logits[:, :-1]으로 targets[:, 1:]을 예측
# clm_loss = ...

# TODO: 유효 학습 토큰 수 비교


### 연습 2: Prefix LM 구현

BERT와 GPT의 장점을 결합한 **Prefix LM** attention mask를 구현하세요.

Prefix LM:
- 앞쪽 `prefix_len`개 토큰은 양방향 (BERT처럼)
- 나머지는 단방향 causal (GPT처럼)

```
seq_len = 6, prefix_len = 3

마스크:
[1, 1, 1, 0, 0, 0]   ← prefix: 양방향 (서로 참조)
[1, 1, 1, 0, 0, 0]
[1, 1, 1, 0, 0, 0]
[1, 1, 1, 1, 0, 0]   ← generation: causal + prefix 참조
[1, 1, 1, 1, 1, 0]
[1, 1, 1, 1, 1, 1]
```

In [ ]:
# TODO: Prefix LM mask 구현
def create_prefix_lm_mask(seq_len, prefix_len):
    """
    Prefix LM attention mask 생성
    - prefix_len 이전: 양방향
    - prefix_len 이후: causal (이전 + prefix 참조 가능)
    """
    # TODO: 구현하세요
    pass

# TODO: mask를 시각화 (plt.imshow)
# TODO: BERT mask, GPT mask와 나란히 비교


---
## 핵심 정리

| 개념 | BERT (Encoder) | GPT (Decoder) | T5 (Enc-Dec) |
|------|---------------|--------------|-------------|
| Attention | 양방향 | 단방향 (Causal) | 양방향 + 단방향 |
| 학습 목표 | MLM (빈칸 채우기) | CLM (다음 토큰) | Seq2Seq |
| 학습 효율 | 15% 토큰만 | 모든 토큰 | task 의존 |
| 강점 | 이해 (분류, NER) | 생성 (대화, 작문) | 변환 (번역, 요약) |
| 약점 | 생성 불가 | 양방향 문맥 부재 | 구조 복잡 |
| 현대 LLM | 거의 안 쓰임 | 주류 (GPT-4, LLaMA) | 일부 (T5, Flan) |

### 왜 Decoder-only가 승리했는가?
1. 학습 효율성 (모든 토큰이 학습 신호)
2. 스케일링 단순성 (하나의 스택)
3. 통합 인터페이스 (모든 task = 다음 토큰 예측)
4. In-context Learning 능력

**다음 노트북**: [05-multi-llm-comparison.ipynb](05-multi-llm-comparison.ipynb) - 멀티 LLM 비교 분석